## `map` function example

The `map(func, seq)` Python function applies the function func to all the elements of the sequence seq. It returns a new list with the elements changed by func

In [ ]:
def f(x):
    return x * x

rdd = [2, 6, -3, 7] #rdd -- resilient distributed datasets
res = map(f, rdd )
res  # Res is an iterator

In [ ]:
print(*res)

4 36 9 49


In [ ]:
from operator import mul
rdd1, rdd2 = [2, 6, -3, 7], [1, -4, 5, 3,5]
res = map(mul, rdd1, rdd2 ) # element wise multipication of rdd1 and rdd2

In [ ]:
print(*res)

2 -24 -15 21


![MapReduce](https://github.com/pnavaro/big-data/blob/master/notebooks/images/mapreduce.jpg?raw=1)

## `functools.reduce` example

The function `reduce(func, seq)` continually applies the function func() to the sequence seq and return a single value. For example, reduce(f, [1, 2, 3, 4, 5]) calculates f(f(f(f(1,2),3),4),5).

In [ ]:
from functools import reduce
from operator import add
rdd = list(range(1,6))
reduce(add, rdd) # computes [1,2,3,4,5]

15

In [ ]:
reduce(mul,rdd) #(((1*2)*3)*4)*5

120

In [ ]:
print(*res)

## Weighted mean and Variance

If the generator of random variable $X$ is discrete with probability mass function $x_1 \mapsto p_1, x_2 \mapsto p_2, \ldots, x_n \mapsto p_n$ then

$$\operatorname{Var}(X) = \left(\sum_{i=1}^n p_i x_i ^2\right) - \mu^2,$$

where $\mu$ is the average value, i.e.

$$\mu = \sum_{i=1}^n p_i x_i. $$

In [ ]:
X = [5, 1, 2, 3, 1, 2, 5, 4]
P = [0.05, 0.05, 0.15, 0.05, 0.15, 0.2, 0.1, 0.25]


### Exercise 5.1

- Write functions to compute the average value and variance using for loops

In [ ]:
mean = reduce(add,map(mul,P,X))
mean_2=mean*mean
X_2=map(mul,X,X)
px=reduce(add,map(mul,P,X_2))
wt_mean=px-mean_2

In [ ]:
wt_mean

1.9600000000000017

In [ ]:
mean_2

7.839999999999999

In [ ]:

X = [5, 1, 2, 3, 1, 2, 5, 4]
P = [0.05, 0.05, 0.15, 0.05, 0.15, 0.2, 0.1, 0.25]


In [ ]:
def variance(P, X):
    mean = sum(p * x for p, x in zip(P, X))
    second_moment = sum(p * x**2 for p, x in zip(P, X))
    return second_moment - mean**2

print(variance(P, X))

1.9600000000000017


In [ ]:
variance(P,X)

1.9600000000000017

In [ ]:
def wt_mean(p,x):
  mean=0
  px=0
  for i in range(len(p)):
    mean=mean+(p[i]*x[i])
    px=px+(p[i]*x[i]*x[i])
  return px-(mean*mean)

In [ ]:
wt_mean(P,X)

1.9600000000000017


### Exercise 5.2

- Write functions to compute the average value and variance using `map` and `reduce`

In [ ]:
a=reduce(add,map(mul,P,X))

In [ ]:
print(a)

2.8


In [ ]:
def wt_mean1(P,X):
  return reduce(add,(map(mul,X,P)))

In [ ]:
wt_mean(P,X)

1.9600000000000017

*NB: Exercises above are just made to help to understand map-reduce process.
This is a bad way to code a variance in Python. You should use [Numpy](http://www.numpy.org) instead.*

## Wordcount

We will modify the `wordcount` application into a map-reduce process.

The `map` process takes text files as input and breaks it into words. The `reduce`  process sums the counts for each word and emits a single key/value with the word and sum.

We need to split the wordcount function we wrote in notebook 04 in order to use map and reduce.

In the following exercices we will implement in Python the Java example described in [Hadoop documentation](https://hadoop.apache.org/docs/current/hadoop-mapreduce-client/hadoop-mapreduce-client-core/MapReduceTutorial.html#Example:_WordCount_v1.0).

In [ ]:
file=open('abx.txt','w')
file.write("this is the content for this file")

33

In [ ]:
file.close()

In [ ]:
file=open('/content/abx.txt','r')
file.read()

'this is the content for this file'

In [ ]:
file.read()

''

In [ ]:
file.seek(0)

0

In [ ]:
file.read()

'this is the content for this file'

In [ ]:
file.seek(0)

0

In [ ]:
a=file.read()

In [ ]:
a

'this is the content for this file'

In [ ]:
type(a)

str

In [ ]:
a.split(' ')

['this', 'is', 'the', 'content', 'for', 'this', 'file']

In [ ]:
file.close()

In [ ]:
%%writefile sample.txt
this is sentence 1
this is sentence 2
this is sentence 3

Overwriting sample.txt


In [ ]:

def token(file):
  a=open(file,'r')
  b=a.read()
  return b.split()

In [ ]:
token('sample.txt')

['this',
 'is',
 'sentence',
 '1',
 'this',
 'is',
 'sentence',
 '2',
 'this',
 'is',
 'sentence',
 '3']

In [ ]:
#read a file ---
#tokenise the words
#create a dictionary of tokens--{'this':[1,1,1],'is':[1,1,1],'a':[1,1,1],'sentence':[1,1,1],'1':[1],'}
#return dictionary with token counts{this:3,is:3,sentence:3,1:1,2:1,3:1}


In [ ]:
file=open('sample.txt','r')

In [ ]:
content=file.read()

In [ ]:
content

'this is sentence 1\nthis is sentence 2\nthis is sentence 3\n'

In [ ]:
file.seek(0)

0

In [ ]:
lines=file.readlines()

In [ ]:
lines

['this is sentence 1\n', 'this is sentence 2\n', 'this is sentence 3\n']

In [ ]:
lines[0]

'this is sentence 1\n'

In [ ]:
a=[]
for i in lines:
  a.extend(i.split())

In [ ]:
a # --{'this':[1,1,1],'is':[1,1,1],'a':[1,1,1],'sentence':[1,1,1],'1':[1],'}

['this',
 'is',
 'sentence',
 '1',
 'this',
 'is',
 'sentence',
 '2',
 'this',
 'is',
 'sentence',
 '3']

In [ ]:
d={}
for i in a:
  if i in d.keys():
    d[i].append(1)
  else:
    d[i]=[1]

In [ ]:
d

{'this': [1, 1, 1],
 'is': [1, 1, 1],
 'sentence': [1, 1, 1],
 '1': [1],
 '2': [1],
 '3': [1]}

In [ ]:
from operator import add

In [ ]:
len(list(d.keys()))

6

In [ ]:
d[list(d.keys())[0]]

[1, 1, 1]

In [ ]:
from functools import reduce
from operator import add

def red(l1):
    return reduce(add, l1)

In [ ]:
for i in range(len(list(d.keys()))):
  d[list(d.keys())[i]]=red(d[list(d.keys())[i]])

In [ ]:
d

{'this': 3, 'is': 3, 'sentence': 3, '1': 1, '2': 1, '3': 1}

In [ ]:
def red(l1):
  return reduce(add,l1)

In [ ]:
from operator import add

In [ ]:
def map_reduce(file):
  a=open(file,'r')
  l1=a.readlines()
  tokens=[]
  for i in l1:
    tokens.extend(i.split())
  d={}
  for i in tokens:
    if i in d.keys():
      d[i].append(1)
    else:
      d[i]=[1]
  key=list(d.keys())
  for i in key:
    d[i]=reduce(add,d[i])
  return d

In [ ]:
map_reduce('sample00.txt')

{'Velit': 1,
 'magnam': 2,
 'non': 8,
 'dolore': 5,
 'sed': 6,
 'dolorem.': 4,
 'Porro': 2,
 'numquam': 3,
 'velit': 5,
 'quaerat.': 2,
 'Tempora': 3,
 'adipisci': 6,
 'porro': 8,
 'quaerat': 4,
 'ipsum': 4,
 'porro.': 2,
 'Voluptatem': 3,
 'aliquam': 2,
 'amet.': 2,
 'Magnam': 1,
 'etincidunt': 4,
 'quisquam': 9,
 'labore.': 1,
 'Aliquam': 2,
 'voluptatem': 5,
 'dolor': 4,
 'dolore.': 3,
 'sit': 5,
 'etincidunt.': 1,
 'Sed': 1,
 'tempora': 5,
 'quiquia': 7,
 'labore': 4,
 'Dolorem': 2,
 'eius': 5,
 'Est': 1,
 'neque': 6,
 'numquam.': 1,
 'Quaerat': 1,
 'est': 4,
 'Etincidunt': 1,
 'est.': 1,
 'Ut': 1,
 'modi': 4,
 'sed.': 1,
 'Amet': 1,
 'consectetur': 4,
 'dolor.': 2,
 'Numquam': 2,
 'dolorem': 4,
 'Dolore': 1,
 'ut.': 1,
 'Sit': 2,
 'ipsum.': 1,
 'Ipsum': 2,
 'sit.': 1,
 'velit.': 1,
 'quiquia.': 1,
 'neque.': 1,
 'adipisci.': 1,
 'Eius': 2,
 'consectetur.': 1,
 'aliquam.': 1}

## Map - Read file and return a key/value pairs

### Exercise 5.3

Write a function `mapper` with a single file name as input that returns a sorted sequence of tuples (word, 1) values.

```pybt
mapper('sample.txt')
[('adipisci', 1), ('adipisci', 1), ('adipisci', 1), ('adipisci', 1), ('adipisci', 1), ('adipisci', 1), ('adipisci', 1), ('aliquam', 1), ('aliquam', 1), ('aliquam', 1), ('aliquam', 1), ('aliquam', 1), ('aliquam', 1), ('aliquam', 1), ('amet', 1), ('amet', 1), ('amet', 1)...
```

## Partition

### Exercise 5.4

Create a function named `partitioner` that stores the key/value pairs from `mapper`  that group (word, 1) pairs into a list as:
```python
partitioner(mapper('sample.txt'))
[('adipisci', [1, 1, 1, 1, 1, 1, 1]), ('aliquam', [1, 1, 1, 1, 1, 1, 1]), ('amet', [1, 1, 1, 1],...]
```

## Reduce - Sums the counts and returns a single key/value (word, sum).

### Exercice 5.5

Write the function `reducer` that read a tuple `(word,[1,1,1,..,1])` and sum the occurrences of word to a final count, and then output the tuple (word,occurences).

```python
reducer(('hello',[1,1,1,1,1])
('hello',5)
```

## Process several files

Let's create 8 files `sample[0-7].txt`. Set most common words at the top of the output list.

In [ ]:
!pip install lorem

In [ ]:
from lorem import text
for i in range(1):
    with open("sample{0:02d}.txt".format(i), "w") as f:
        f.write(text())

In [ ]:
import glob
files = sorted(glob.glob('sample0*.txt'))
files

['sample00.txt']

### Exercise 5.6
- Use functions implemented above to count (word, occurences) by using a for loops over files and partitioned data.

### Exercise 5.7
- This time use `map` function to apply mapper and reducer.